# 실습 1. Python 텍스트 데이터 처리 (Day 1 / 90분)

> 시나리오: **쇼핑몰 리뷰 데이터 정제** — `data/reviews.csv`
>
> Day 1(Pandas·정규식)은 이 저장소의 **로컬 노트북**으로 진행한다.

## Task
1. CSV 로드 후 **결측·이상치 리포트** 작성
2. `text` 컬럼 정제 — URL/특수문자/공백 정규화
3. **정규식**으로 광고 패턴(`[광고]`, `^협찬`, `구매처:`) 제거
4. `rating` 별 평균 텍스트 길이 / 단어 수 집계
5. 정제 결과를 `reviews_clean.parquet` 으로 저장

In [1]:
import re
# 이메일 추출
text = "abc@gmail.com hello world 010-557-8888"
emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
print("이메일:", emails)

이메일: ['abc@gmail.com']


In [2]:
import pandas as pd

# data/reviews.csv 가 없으면: 터미널에서 `python data/make_reviews.py`
df = pd.read_csv("../data/reviews.csv")
print(df.shape)
df.head()

(10033, 5)


,id,user,text,rating,created_at
0,1,user087,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,2.0,2026/06/18 02:37
1,2,user072,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,4.0,23.02.2026
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14T10:17:00
3,4,user045,[광고] 지금 구매하면 사은품 증정! 포장도 꼼꼼하고 품질이 기대 이상이에요 😡,4.0,NaN
4,5,user047,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,5.0,2026-05-07


## 1) 결측·이상치 리포트

In [3]:
df.info()
print("\n결측치:\n", df.isna().sum())
print("\nrating 분포:\n", df["rating"].value_counts(dropna=False))

# created_at 은 형식이 제각각 → errors='coerce' 로 파싱 실패는 NaT 처리
df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", format="mixed")
print("\n날짜 파싱 실패(NaT):", df["created_at"].isna().sum(), "건")
# TODO: NaT 행을 버릴지/보존할지 결정
df = df.dropna(subset=["created_at"])
print("\ncreated_at dtype:", df["created_at"].dtype)
print("\ncreated_at min/max:", df["created_at"].min(), "/", df["created_at"].max())


<class 'pandas.DataFrame'>
RangeIndex: 10033 entries, 0 to 10032
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          10033 non-null  int64  
 1   user        10033 non-null  str    
 2   text        10033 non-null  str    
 3   rating      9527 non-null   float64
 4   created_at  8033 non-null   str    
dtypes: float64(1), int64(1), str(3)
memory usage: 1.3 MB

결측치:
 id               0
user             0
text             0
rating         506
created_at    2000
dtype: int64

rating 분포:
 rating
4.0    2387
5.0    2363
1.0    1715
2.0    1669
3.0    1392
NaN     506
0.0       1
Name: count, dtype: int64

날짜 파싱 실패(NaT): 2000 건

created_at dtype: datetime64[us]

created_at min/max: 2026-01-01 00:00:00 / 2026-12-06 00:00:00


## 2) 텍스트 정제 파이프라인 (슬라이드 `clean()` 참고)

`.str` 로 컬럼 전체를 한 번에 정제한다. 정규식 패턴 풀이:
- `https?://\S+` → http/https 로 시작하는 URL (공백 전까지)
- `[^\w가-힣\s]` → 단어문자(`\w`)·한글(`가-힣`)·공백(`\s`) **이 아닌** 것 = 특수문자/이모지
- `\s+` → 연속 공백을 1칸으로

In [4]:
import re

def clean(s: pd.Series) -> pd.Series:
    return (
        s.fillna("")
         .str.strip()
         .str.replace(r"https?://\S+", "", regex=True)        # URL 제거
         .str.replace(r"[^\w가-힣\s]", " ", regex=True)        # 특수문자 제거 (가-힣 = U+AC00~U+D7A3)
         .str.replace(r"\s+", " ", regex=True)                 # 공백 정규화
         .str.strip()
    )

df["clean_text"] = clean(df["text"])
df[["text", "clean_text"]].head()

,text,clean_text
0,협찬 받아 작성한 후기입니다. 생각보다 너무 작고 부실해요 👍,협찬 받아 작성한 후기입니다 생각보다 너무 작고 부실해요
1,협찬 받아 작성한 후기입니다. 배송이 정말 빨라서 깜짝 놀랐어요 👍,협찬 받아 작성한 후기입니다 배송이 정말 빨라서 깜짝 놀랐어요
2,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,배송이 일주일이나 걸렸습니다 별로
4,[광고] 지금 구매하면 사은품 증정! 배송이 정말 빨라서 깜짝 놀랐어요 ㅠㅠ,광고 지금 구매하면 사은품 증정 배송이 정말 빨라서 깜짝 놀랐어요
5,[광고] 지금 구매하면 사은품 증정! 가성비 최고입니다 또 살게요 👍,광고 지금 구매하면 사은품 증정 가성비 최고입니다 또 살게요


## 3) 광고 패턴 제거 (정규식)

In [5]:
AD = re.compile(r"\[광고\]|^협찬|구매처\s*:")
is_ad = df["text"].str.contains(AD)
print(f"광고성 리뷰: {is_ad.sum()}건")
# TODO: 광고 문구만 제거할지, 행 자체를 버릴지 결정하고 처리
df = df[~is_ad].copy()
print(f"광고성 리뷰 제거 후: {len(df)}건")
df.head()


광고성 리뷰: 3942건
광고성 리뷰 제거 후: 4091건


,id,user,text,rating,created_at,clean_text
2,3,user021,배송이 일주일이나 걸렸습니다 별로 ㅠㅠ,2.0,2026-06-14 10:17:00,배송이 일주일이나 걸렸습니다 별로
6,7,user069,가성비 최고입니다 또 살게요,5.0,2026-08-06 00:00:00,가성비 최고입니다 또 살게요
7,8,user005,사이즈도 딱 맞고 마감이 깔끔해요 🔥,4.0,2026-03-13 08:04:00,사이즈도 딱 맞고 마감이 깔끔해요
9,10,user029,재구매 의사 100% 강추합니다 😡,5.0,2026-02-17 00:00:00,재구매 의사 100 강추합니다
10,11,user050,생각보다 너무 작고 부실해요 😍 https://example.com/event,1.0,2026-04-20 14:33:00,생각보다 너무 작고 부실해요


## 4) rating 별 텍스트 길이 / 단어 수 집계

In [6]:
df["length"] = df["clean_text"].str.len()
df["words"] = df["clean_text"].str.split().str.len()
df.groupby("rating")[["length", "words"]].mean()

,length,words
rating,,
0.0,16.000000,4.000000
1.0,18.385915,4.884507
2.0,18.229018,4.830725
3.0,14.261986,3.945205
4.0,17.815401,4.497890
5.0,17.891693,4.503680


In [7]:
df.info()

<class 'pandas.DataFrame'>
Index: 4091 entries, 2 to 10032
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   id          4091 non-null   int64         
 1   user        4091 non-null   str           
 2   text        4091 non-null   str           
 3   rating      3897 non-null   float64       
 4   created_at  4091 non-null   datetime64[us]
 5   clean_text  4091 non-null   str           
 6   length      4091 non-null   int64         
 7   words       4091 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(3), str(3)
memory usage: 728.2 KB


## 5) 저장 — 실습 2의 입력이 된다

In [8]:
df.to_parquet("../data/reviews_clean.parquet", index=False)
print("saved: data/reviews_clean.parquet")

saved: data/reviews_clean.parquet
